In [ ]:
# PART 1 - SAXPY
# Clone repository
!git clone https://github.com/stanford-cs149/asst3


Cloning into 'asst3'...
remote: Enumerating objects: 602, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 602 (delta 83), reused 77 (delta 77), pack-reused 503 (from 1)
Receiving objects: 100% (602/602), 14.99 MiB | 17.54 MiB/s, done.
Resolving deltas: 100% (318/318), done.


In [ ]:
# Build and run SAXPY
!cd /content/asst3/saxpy && make && ./cudaSaxpy

mkdir -p objs/
g++ -m64 -O3 -Wall -o cudaSaxpy objs/main.o  objs/saxpy.o -L/usr/local/cuda/lib64/ -lcudart
---------------------------------------------------------
Found 1 CUDA devices
Device 0: Tesla T4
   SMs:        40
   Global mem: 14913 MB
   CUDA Cap:   7.5
---------------------------------------------------------
Running 3 timing tests:
Effective BW by CUDA saxpy: 272.413 ms		[4.103 GB/s]
Kernel-only time:           4.805 ms		[232.586 GB/s]
Effective BW by CUDA saxpy: 298.750 ms		[3.741 GB/s]
Kernel-only time:           4.643 ms		[240.729 GB/s]
Effective BW by CUDA saxpy: 288.957 ms		[3.868 GB/s]
Kernel-only time:           4.702 ms		[237.685 GB/s]


In [ ]:
# PART 2 - PARALLEL PREFIX SUM
# Build scan
%cd /content/asst3/scan


/content/asst3/scan


In [ ]:
!make
# Test scan correctness at all sizes
!./cudaScan -n 1000000
!./cudaScan -n 10000000
!./cudaScan -n 20000000
!./cudaScan -n 40000000
# Test find_repeats
!./cudaScan -m find_repeats -n 1000000
!python3 checker.py scan
!python3 checker.py find_repeats

mkdir -p objs/
g++ -m64 -O3 -Wall -o cudaScan objs/main.o  objs/scan.o -L/usr/local/cuda/lib64/ -lcudart
---------------------------------------------------------
Found 1 CUDA devices
Device 0: Tesla T4
   SMs:        40
   Global mem: 14913 MB
   CUDA Cap:   7.5
---------------------------------------------------------
Array size: 1000000
Student GPU time: 0.262 ms
Scan outputs are correct!
---------------------------------------------------------
Found 1 CUDA devices
Device 0: Tesla T4
   SMs:        40
   Global mem: 14913 MB
   CUDA Cap:   7.5
---------------------------------------------------------
Array size: 10000000
Student GPU time: 0.916 ms
Scan outputs are correct!
---------------------------------------------------------
Found 1 CUDA devices
Device 0: Tesla T4
   SMs:        40
   Global mem: 14913 MB
   CUDA Cap:   7.5
---------------------------------------------------------
Array size: 20000000
Student GPU time: 1.855 ms
Scan outputs are correct!
-----------------------

In [ ]:
# PART 3 - CIRCLE RENDERER
# Install OpenGL dependencies
!apt-get install -y freeglut3-dev libglu1-mesa-dev
!ln -sf /usr/lib/x86_64-linux-gnu/libGLX_mesa.so.0 /usr/lib/x86_64-linux-gnu/libGL.so

# Fix Makefile to link correctly
%cd /content/asst3/render

# Download original cudaRenderer.cu
!curl -o /content/asst3/render/cudaRenderer.cu \
  https://raw.githubusercontent.com/stanford-cs149/asst3/master/render/cudaRenderer.cu

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libglu1-mesa-dev is already the newest version (9.0.2-1).
freeglut3-dev is already the newest version (2.8.1-6).
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.
/content/asst3/render
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 21413  100 21413    0     0  58250      0 --:--:-- --:--:-- --:--:-- 58187


In [ ]:
# Patch cudaRenderer.cu with correct tile-based kernel
with open('/content/asst3/render/cudaRenderer.cu', 'r') as f:
    content = f.read()

new_kernel = '''
#define TILE_DIM 16

__global__ void kernelRenderTiles() {

    int tileStartX = blockIdx.x * TILE_DIM;
    int tileStartY = blockIdx.y * TILE_DIM;
    int pixelX = tileStartX + threadIdx.x;
    int pixelY = tileStartY + threadIdx.y;
    int imageWidth  = cuConstRendererParams.imageWidth;
    int imageHeight = cuConstRendererParams.imageHeight;

    if (pixelX >= imageWidth || pixelY >= imageHeight) return;

    float invW = 1.f / imageWidth;
    float invH = 1.f / imageHeight;
    float tileL = tileStartX * invW;
    float tileR = (tileStartX + TILE_DIM) * invW;
    float tileB = tileStartY * invH;
    float tileT = (tileStartY + TILE_DIM) * invH;

    float2 pixelCenter = make_float2(invW*(pixelX+0.5f), invH*(pixelY+0.5f));
    float4* imgPtr = (float4*)(&cuConstRendererParams.imageData[4*(pixelY*imageWidth+pixelX)]);
    float4 pixelColor = *imgPtr;

    int numCircles = cuConstRendererParams.numCircles;
    for (int circleIdx = 0; circleIdx < numCircles; circleIdx++) {
        int idx3 = 3 * circleIdx;
        float cx  = cuConstRendererParams.position[idx3];
        float cy  = cuConstRendererParams.position[idx3+1];
        float rad = cuConstRendererParams.radius[circleIdx];
        float nearX = fmaxf(tileL, fminf(cx, tileR));
        float nearY = fmaxf(tileB, fminf(cy, tileT));
        float dx = cx - nearX, dy = cy - nearY;
        if (dx*dx + dy*dy > rad*rad) continue;
        float3 p = *(float3*)(&cuConstRendererParams.position[idx3]);
        shadePixel(circleIdx, pixelCenter, p, &pixelColor);
    }
    *imgPtr = pixelColor;
}
'''

content = content.replace('CudaRenderer::CudaRenderer() {',
    new_kernel + '\n////////////////////////////////////////////////////////////////////////////////////////\n\nCudaRenderer::CudaRenderer() {')

content = content.replace(
    '    // 256 threads per block is a healthy number\n    dim3 blockDim(256, 1);\n    dim3 gridDim((numCircles + blockDim.x - 1) / blockDim.x);',
    '    dim3 blockDim(TILE_DIM, TILE_DIM);\n    dim3 gridDim((image->width+TILE_DIM-1)/TILE_DIM, (image->height+TILE_DIM-1)/TILE_DIM);'
)
content = content.replace(
    'kernelRenderCircles<<<gridDim, blockDim>>>();',
    'kernelRenderTiles<<<gridDim, blockDim>>>();'
)

with open('/content/asst3/render/cudaRenderer.cu', 'w') as f:
    f.write(content)

print('Patch applied successfully')

Patch applied successfully


In [ ]:
# Build and test all 8 scenes
%cd /content/asst3/render
!make clean && make
!./render -r cuda rgb -c
!./render -r cuda rand10k -c
!./render -r cuda rand100k -c
!./render -r cuda pattern -c
!./render -r cuda snowsingle -c
!./render -r cuda biglittle -c
!./render -r cuda rand1M -c
!./render -r cuda micro2M -c

/content/asst3/render
rm -rf objs *~ render logs
mkdir -p objs/
g++ -m64 main.cpp -O3 -Wall -g -c -o objs/main.o
g++ -m64 display.cpp -O3 -Wall -g -c -o objs/display.o
nvcc benchmark.cu -O3 -m64 -c -o objs/benchmark.o
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
g++ -m64 refRenderer.cpp -O3 -Wall -g -c -o objs/refRenderer.o
nvcc cudaRenderer.cu -O3 -m64 -c -o objs/cudaRenderer.o
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
g++ -m64 noise.cpp -O3 -Wall -g -c -o objs/noise.o
g++ -m64 ppm.cpp -O3 -Wall -g -c -o objs/ppm.o
g++ -m64 sceneLoader.cpp -O3 -Wall -g -c -o objs/sceneLoader.o
g++ -m64 -O3 -Wall -g -o render objs/main.o objs/display.o objs/benchmark.o objs/refRenderer.o objs/cudaRenderer.o objs/noise.o 